In [4]:
"""
agropulse_xai/run_pipeline_harmonizer_v3_360px_STRICT.py

Script del rol Matemático adaptado a la delimitación exacta de la parcela agrícola (Fase 1):
- LECTURA ESTRICTA: Solo utiliza la carpeta parquet_VERIFICADO (sin fallbacks a crudos).
- Conservación de historial de cultivos (crop_2016 a crop_2025).
- Uso de variables edáficas directamente desde la grilla.
- Inyección de error estructural físico (sigma_smap_m3m3).
"""

import os
import json
import numpy as np
import pandas as pd
from typing import Dict, Tuple

# ==============================================================================
# 1. CONEXIÓN A GOOGLE DRIVE Y RUTAS (MODO ESTRICTO)
# ==============================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/agropulse_xai'
    print("[INFO] Montado Google Drive en Colab correctamente.")
except ImportError:
    BASE_DIR = os.path.abspath(os.path.dirname(__file__))
    print("[INFO] Ejecutando en entorno local.")

# Única fuente de verdad permitida
DATA_DIR_VERIF   = os.path.join(BASE_DIR, 'data', 'parquet_VERIFICADO')

DATA_DIR_OUTPUT  = os.path.join(
    BASE_DIR,
    'RUTA_FINAL',
    '1_TRABAJO_EQUIPO',
    'RESULTADOS A REVISAR (MATEMÁTICO)'
)
os.makedirs(DATA_DIR_OUTPUT, exist_ok=True)

# Rutas exactas requeridas
PATH_GRID_OFICIAL = os.path.join(DATA_DIR_VERIF, "grid_30m.parquet")
PATH_ALIGNED      = os.path.join(DATA_DIR_VERIF, "aligned_2016_2025.parquet")
PATH_EVAL         = os.path.join(DATA_DIR_VERIF, "evaluated_metrics.parquet")

# Archivos de salida
PATH_OUT_GRID   = os.path.join(DATA_DIR_OUTPUT, "harmonized_grid_30m.parquet")
PATH_OUT_TOPSIS = os.path.join(DATA_DIR_OUTPUT, "topsis_rankings.parquet")
PATH_OUT_META   = os.path.join(DATA_DIR_OUTPUT, "metadata_run.json")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Montado Google Drive en Colab correctamente.


In [5]:
# ==============================================================================
# 2. FUNCIONES MATEMÁTICAS Y ESTADÍSTICAS
# ==============================================================================

def downscale_smap_rootzone_30m_v2(
    df_spatial_30m: pd.DataFrame, smap_val_9km: float, gamma_decay: float = 0.5
) -> Tuple[pd.DataFrame, float, str]:
    df = df_spatial_30m.copy()
    awc_col = 'ssurgo_awc'
    if awc_col not in df.columns:
        raise ValueError("Falta la columna de capacidad de retención de agua (AWC).")

    df[awc_col] = df[awc_col].fillna(df[awc_col].median())

    awc_mean_global = df[awc_col].mean()
    r_ratio_global = df[awc_col] / awc_mean_global
    sm_30m_global = smap_val_9km * (r_ratio_global ** gamma_decay)
    df['smap_sm_30m'] = np.clip(sm_30m_global, 0.01, 0.55)

    n_blocks = 4
    block_size = len(df) // n_blocks
    errors_cv = []

    for k in range(n_blocks):
        idx_test = range(k * block_size, (k + 1) * block_size if k < n_blocks - 1 else len(df))
        idx_train = [i for i in range(len(df)) if i not in idx_test]

        awc_mean_train = df.iloc[idx_train][awc_col].mean()
        r_ratio_test = df.iloc[idx_test][awc_col] / awc_mean_train
        sm_30m_test = smap_val_9km * (r_ratio_test ** gamma_decay)
        errors_cv.extend(sm_30m_test - smap_val_9km)

    rmse_cv = float(np.sqrt(np.mean(np.square(errors_cv))))
    sigma_smap_m3m3 = max(rmse_cv, 0.04)
    df['sigma_smap_m3m3'] = sigma_smap_m3m3
    return df, sigma_smap_m3m3, "MEDIA"


def lapse_rate_correction_temperature(
    t_interp: float, z_30m: np.ndarray, z_coarse: float, gamma_lapse: float = 0.0065
) -> np.ndarray:
    return t_interp - gamma_lapse * (z_30m - z_coarse)


def run_topsis_dirichlet_v2(
    df_evaluated: pd.DataFrame, weights_dict: Dict[str, float], n_sim: int = 200, concentration: float = 15.0, seed: int = 42
) -> pd.DataFrame:
    np.random.seed(seed)
    possible_criteria = ['water_stress_index', 'soil_loss_rusle', 'nitrogen_balance', 'profitability_usd', 'pest_penalty', 'nitrogen_deficit']
    criteria_cols = [c for c in possible_criteria if c in df_evaluated.columns]

    if len(criteria_cols) < 3:
        raise ValueError(f"Insuficientes criterios para TOPSIS. Disponibles: {criteria_cols}")

    is_benefit = {'water_stress_index': False, 'soil_loss_rusle': False, 'nitrogen_balance': True, 'profitability_usd': True, 'pest_penalty': False, 'nitrogen_deficit': False}
    weight_keys = {'water_stress_index': 'water', 'soil_loss_rusle': 'erosion', 'nitrogen_balance': 'nitrogen', 'profitability_usd': 'profit', 'pest_penalty': 'pest', 'nitrogen_deficit': 'nitrogen'}

    df = df_evaluated.copy()
    raw_weights = np.array([weights_dict.get(weight_keys[c], 1.0) for c in criteria_cols], dtype=float)
    base_p = raw_weights / np.sum(raw_weights)
    alpha = base_p * concentration
    sim_weights = np.random.dirichlet(alpha, size=n_sim)

    def process_topsis_group(group: pd.DataFrame) -> pd.DataFrame:
        X = group[criteria_cols].values.astype(float)
        m = len(X)
        norm_factors = np.sqrt(np.sum(X**2, axis=0))
        norm_factors[norm_factors == 0] = 1.0
        R = X / norm_factors
        scores_sim = np.zeros((n_sim, m))

        for k in range(n_sim):
            w_k = sim_weights[k]
            V = R * w_k
            A_plus, A_minus = np.zeros(len(criteria_cols)), np.zeros(len(criteria_cols))
            for j, col in enumerate(criteria_cols):
                if is_benefit[col]:
                    A_plus[j], A_minus[j] = np.max(V[:, j]), np.min(V[:, j])
                else:
                    A_plus[j], A_minus[j] = np.min(V[:, j]), np.max(V[:, j])
            S_plus = np.sqrt(np.sum((V - A_plus)**2, axis=1))
            S_minus = np.sqrt(np.sum((V - A_minus)**2, axis=1))
            denom = S_plus + S_minus
            denom[denom == 0] = 1e-9
            scores_sim[k, :] = S_minus / denom

        mean_scores = np.mean(scores_sim, axis=0)
        res = group.copy()
        res['topsis_score'] = mean_scores
        res['rank'] = pd.Series(mean_scores, index=group.index).rank(ascending=False, method='min').astype(int)
        return res

    if 'pixel_id' in df.columns and df['pixel_id'].nunique() > 1:
        df_ranked = df.groupby('pixel_id', group_keys=False).apply(process_topsis_group)
    else:
        df_ranked = process_topsis_group(df)
    return df_ranked.sort_values(by=['pixel_id', 'rank'])

In [6]:
# ==============================================================================
# 3. PIPELINE DE EJECUCIÓN
# ==============================================================================

def main():
    print("=== INICIANDO PIPELINE DE ARMONIZACIÓN V3 (MODO ESTRICTO: VERIFICADO) ===")

    # 1. Búsqueda estricta de Grilla Base VERIFICADA
    if not os.path.exists(PATH_GRID_OFICIAL):
        raise FileNotFoundError(f"Error crítico: No se encontró la grilla base verificada en: {PATH_GRID_OFICIAL}")

    df_grid_raw = pd.read_parquet(PATH_GRID_OFICIAL)
    print(f"-> Carga inicial desde: {PATH_GRID_OFICIAL} ({len(df_grid_raw)} píxeles).")

    # Filtrado estricto a la parcela (360 píxeles)
    valid_pixel_ids = set([f"px_{i:04d}" for i in range(360)])
    df_grid = df_grid_raw[df_grid_raw['pixel_id'].isin(valid_pixel_ids)].copy()

    # 2. Rescatar historial de cultivos (crop_2016 a crop_2025)
    crop_cols = [c for c in df_grid.columns if c.startswith('crop_')]
    if len(crop_cols) > 0:
        print(f"-> Historial de cultivos detectado correctamente ({len(crop_cols)} años).")
    else:
        print("-> [ADVERTENCIA] No se detectaron columnas de cultivos (crop_YYYY) en la grilla base.")

    # 3. Tratamiento de variable de suelos (AWC)
    if 'agua_disponible_awc' in df_grid.columns:
        df_grid.rename(columns={'agua_disponible_awc': 'ssurgo_awc'}, inplace=True)
        print("-> Variable de retención de agua edáfica ajustada.")
    else:
        raise KeyError("La grilla base no contiene la columna 'agua_disponible_awc' requerida para el downscaling.")

    # 4. Carga de Series Hídricas y Climáticas (Solo datos verificados)
    smap_mean, tmax_base, gpm_mean = 0.25, 15.0, 0.0
    if os.path.exists(PATH_ALIGNED):
        df_aligned = pd.read_parquet(PATH_ALIGNED)
        if 'smap_rootzone' in df_aligned.columns: smap_mean = float(df_aligned['smap_rootzone'].dropna().mean())
        if 'tmax_c' in df_aligned.columns: tmax_base = float(df_aligned['tmax_c'].mean())
        if 'gpm_rain_mm' in df_aligned.columns: gpm_mean = float(df_aligned['gpm_rain_mm'].mean())
        print("-> Carga exitosa de series climáticas verificadas.")
    else:
        print(f"-> [ADVERTENCIA] No se encontró el archivo climático verificado en {PATH_ALIGNED}. Usando medias por defecto.")

    # 5. Downscaling
    df_spatial_30m = df_grid.copy()
    df_harmonized, rmse_spatial, certeza_max = downscale_smap_rootzone_30m_v2(
        df_spatial_30m=df_spatial_30m, smap_val_9km=smap_mean, gamma_decay=0.5
    )
    print(f"-> Downscaling completado. Score ajustado a mínimo estructural (m³/m³): {rmse_spatial:.4f}")

    # 6. Corrección Topográfica
    z_base = df_grid['elevation_m'].mean() if 'elevation_m' in df_grid.columns else 290.0
    z_values = df_grid['elevation_m'].values if 'elevation_m' in df_grid.columns else np.full(len(df_grid), 290.0)

    df_harmonized['power_tmax_30m'] = lapse_rate_correction_temperature(
        t_interp=tmax_base, z_30m=z_values, z_coarse=z_base, gamma_lapse=0.0065
    )
    df_harmonized['gpm_precip_30m'] = gpm_mean

    # 7. Exportar Grilla Armonizada y Metadata
    base_cols_export = [
        'pixel_id', 'lon', 'lat', 'elevation_m', 'slope_pct',
        'ssurgo_awc', 'awc_l', 'awc_h', 'smap_sm_30m', 'gpm_precip_30m',
        'power_tmax_30m', 'sigma_smap_m3m3'
    ]

    cols_export = base_cols_export + crop_cols
    cols_existentes = [c for c in cols_export if c in df_harmonized.columns]

    df_export = df_harmonized[cols_existentes]
    df_export.to_parquet(PATH_OUT_GRID, index=False)
    print(f"\n[OK] Generado exitosamente: {PATH_OUT_GRID} ({len(df_export)} filas × {len(df_export.columns)} columnas)")

    metadata = {
        "nota_auditoria": "El valor 0.04 en sigma_smap_m3m3 entra como piso de error estructural nominal de SMAP ante homogeneidad de suelo en la parcela piloto.",
        "columnas_cultivo_incluidas": crop_cols,
        "n_pixeles": len(df_export)
    }
    with open(PATH_OUT_META, 'w') as f:
        json.dump(metadata, f, indent=4)
    print(f"[OK] Metadata guardada en: {PATH_OUT_META}")

    # 8. Optimización TOPSIS v2
    df_eval = None
    if os.path.exists(PATH_EVAL):
        df_eval_raw = pd.read_parquet(PATH_EVAL)
        if 'pixel_id' in df_eval_raw.columns:
            df_eval = df_eval_raw[df_eval_raw['pixel_id'].isin(valid_pixel_ids)].copy()
        else:
            df_eval = df_eval_raw.copy()

        if df_eval is not None:
            pesos_defecto = {'water': 4.0, 'erosion': 4.0, 'nitrogen': 3.0, 'profit': 4.0, 'pest': 3.0}
            df_topsis = run_topsis_dirichlet_v2(df_eval, pesos_defecto, n_sim=200)
            df_topsis.to_parquet(PATH_OUT_TOPSIS, index=False)
            print(f"[OK] Generado exitosamente TOPSIS v2: {PATH_OUT_TOPSIS}")
    else:
        print(f"[INFO] 'evaluated_metrics.parquet' aún no se encuentra en la carpeta verificada para el TOPSIS.")

if __name__ == "__main__":
    main()

=== INICIANDO PIPELINE DE ARMONIZACIÓN V3 (MODO ESTRICTO: VERIFICADO) ===


FileNotFoundError: Error crítico: No se encontró la grilla base verificada en: /content/drive/MyDrive/agropulse_xai/data/parquet_VERIFICADO/grid_30m.parquet

In [9]:
import os
import pandas as pd
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta de la carpeta verificada
ruta_verificado = '/content/drive/MyDrive/agropulse_xai/data/parquet_VERIFICADO'

# 3. Verificar que la ruta exista y listar los archivos
if not os.path.exists(ruta_verificado):
    print(f"[ERROR] No se encuentra la ruta: {ruta_verificado}")
    print("Asegúrate de haber creado el acceso directo de 'agropulse_xai' en 'Mi unidad'.")
else:
    archivos = [f for f in os.listdir(ruta_verificado) if f.endswith('.parquet')]
    print(f"=== SE ENCONTRARON {len(archivos)} BASES EN 'parquet_VERIFICADO' ===\n")

    for archivo in sorted(archivos):
        ruta_completa = os.path.join(ruta_verificado, archivo)
        df = pd.read_parquet(ruta_completa)

        print("-" * 70)
        print(f"ARCHIVO: {archivo}")
        print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
        print(f"Columnas: {list(df.columns)}")
        print("\nPrimeras 3 filas:")
        print(df.head(3))
        print("-" * 70 + "\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== SE ENCONTRARON 5 BASES EN 'parquet_VERIFICADO' ===

----------------------------------------------------------------------
ARCHIVO: D2_ssurgo_soil.parquet
Dimensiones: 4 filas × 12 columnas
Columnas: ['mukey', 'muname', 'compname', 'comppct_r', 'hzdept_r', 'hzdepb_r', 'materia_organica', 'agua_disponible_awc', 'erodabilidad_k', 'arena', 'limo', 'arcilla']

Primeras 3 filas:
     mukey                                             muname compname  \
0  2765537  Clarion loam, Bemis moraine, 2 to 6 percent sl...  Clarion   
1  2765537  Clarion loam, Bemis moraine, 2 to 6 percent sl...  Clarion   
2  2765537  Clarion loam, Bemis moraine, 2 to 6 percent sl...  Clarion   

   comppct_r  hzdept_r  hzdepb_r  materia_organica  agua_disponible_awc  \
0         85        35        84              0.75                 0.18   
1         85         0        23           